# Appliance Energy Consumption Analysis

This notebook analyzes electricity consumption by appliance/end-use category throughout the year using data from the residential electrification cost modeling system.

## Data Sources:
- **Real Electricity Data**: `electricity_loads_{county}.csv` - Hourly consumption by specific appliances from NREL ResStock
- **Simulated Electricity Data**: `electricity_loads_simulated_{county}.csv` - Electrified appliance consumption (heat pump, induction stove, electric water heater)
- **Gas Data**: `gas_loads_{county}.csv` - Natural gas consumption converted to kWh equivalent for comparison

## End-Use Categories:
Based on analysis of step7_combine_real_and_simulated_electricity_loads.py and step3/step4 processing steps.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")

In [ ]:
# Configuration - modify these as needed
BASE_DIR = Path('data/loadprofiles')
SCENARIO = 'baseline'  # Options: baseline, heat_pump, heat_pump_and_induction_stove, etc.
HOUSING_TYPE = 'single-family-detached'
COUNTY = 'alameda'  # County slug (lowercase, dashes for spaces)

# Define data paths
data_dir = BASE_DIR / SCENARIO / HOUSING_TYPE / COUNTY
electricity_file = data_dir / f'electricity_loads_{COUNTY}.csv'
gas_file = data_dir / f'gas_loads_{COUNTY}.csv'
simulated_file = data_dir / f'electricity_loads_simulated_{COUNTY}.csv'

print(f"Analysis Configuration:")
print(f"  Scenario: {SCENARIO}")
print(f"  County: {COUNTY.title()}")
print(f"  Data directory: {data_dir}")
print(f"  Files exist:")
print(f"    Electricity: {electricity_file.exists()}")
print(f"    Gas: {gas_file.exists()}")
print(f"    Simulated: {simulated_file.exists()}")

In [ ]:
# Define appliance categories based on step3/step4 and step7 analysis
ELECTRICITY_CATEGORIES = {
    "Cooling": ["ceiling_fan"],
    "Appliances": ["clothes_dryer", "dishwasher", "freezer", "refrigerator"],
    "Lighting": ["lighting_garage", "lighting_interior"],
    "Plug Loads": ["plug_loads"],
    "Pool/Spa": ["permanent_spa_heat", "permanent_spa_pump", "pool_heater", "pool_pump"],
    "Other Electric": ["mech_vent"]
}

GAS_CATEGORIES = {
    "Heating": ["heating"],
    "Hot Water": ["hot_water"],
    "Cooking": ["range_oven"],
    "Other Gas": ["clothes_dryer", "fireplace"]
}

SIMULATED_CATEGORIES = {
    "Heat Pump": "simulated.electricity.heat_pump.energy_consumption.electricity.kwh",
    "Induction Cooking": "simulated.electricity.induction_stove.energy_consumption.electricity.kwh",
    "Electric Hot Water": "simulated.electricity.hot_water.energy_consumption.electricity.kwh"
}

# Color mapping for consistent visualization
COLOR_MAP = {
    "Heating": "#FF6B6B",
    "Heat Pump": "#FF8E53",
    "Cooling": "#4ECDC4",
    "Hot Water": "#45B7D1",
    "Electric Hot Water": "#96CEB4",
    "Cooking": "#FFEAA7",
    "Induction Cooking": "#DDA0DD",
    "Appliances": "#FD79A8",
    "Lighting": "#FDCB6E",
    "Plug Loads": "#6C5CE7",
    "Pool/Spa": "#00B894",
    "Other Electric": "#A29BFE",
    "Other Gas": "#E17055"
}

print("Categories defined:")
print(f"  Electricity: {list(ELECTRICITY_CATEGORIES.keys())}")
print(f"  Gas: {list(GAS_CATEGORIES.keys())}")
print(f"  Simulated: {list(SIMULATED_CATEGORIES.keys())}")

In [ ]:
def load_and_process_electricity_data(file_path):
    """
    Load electricity consumption data and aggregate by appliance category.
    Returns DataFrame with hourly consumption by category.
    """
    if not file_path.exists():
        print(f"Warning: File {file_path} not found")
        return None
    
    # Load data
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    print(f"Loaded electricity data: {len(df)} hourly records")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    
    # Create category aggregations
    category_data = {}
    
    for category, appliances in ELECTRICITY_CATEGORIES.items():
        category_consumption = pd.Series(0.0, index=df.index)
        
        for appliance in appliances:
            col_name = f"out.electricity.{appliance}.energy_consumption"
            if col_name in df.columns:
                category_consumption += df[col_name]
                
        if category_consumption.sum() > 0:
            category_data[category] = category_consumption
    
    return pd.DataFrame(category_data)

def load_and_process_gas_data(file_path):
    """
    Load gas consumption data and aggregate by appliance category.
    Returns DataFrame with hourly consumption by category (converted to kWh equivalent).
    """
    if not file_path.exists():
        print(f"Warning: File {file_path} not found")
        return None
    
    # Load data - gas data is 15-minute intervals
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # Resample to hourly (sum 15-minute intervals)
    df = df.resample('H').sum()
    
    print(f"Loaded gas data: {len(df)} hourly records (resampled from 15-min)")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    
    # Create category aggregations using building average kWh columns
    category_data = {}
    
    for category, appliances in GAS_CATEGORIES.items():
        category_consumption = pd.Series(0.0, index=df.index)
        
        for appliance in appliances:
            col_name = f"out.natural_gas.{appliance}.energy_consumption.gas.building_avg.kwh"
            if col_name in df.columns:
                category_consumption += df[col_name]
                
        if category_consumption.sum() > 0:
            category_data[category] = category_consumption
    
    return pd.DataFrame(category_data)

def load_and_process_simulated_data(file_path, scenario):
    """
    Load simulated electricity consumption data for electrified appliances.
    Returns DataFrame with hourly consumption by simulated appliance category.
    """
    if not file_path.exists():
        print(f"Warning: File {file_path} not found")
        return None
    
    # Load data - simulated data is 15-minute intervals
    df = pd.read_csv(file_path, parse_dates=['timestamp'])
    df.set_index('timestamp', inplace=True)
    
    # Resample to hourly (sum 15-minute intervals)
    df = df.resample('H').sum()
    
    print(f"Loaded simulated data: {len(df)} hourly records (resampled from 15-min)")
    print(f"Available columns: {list(df.columns)}")
    
    # Create category aggregations - only include relevant categories for scenario
    category_data = {}
    
    for category, col_name in SIMULATED_CATEGORIES.items():
        if col_name in df.columns:
            consumption = df[col_name]
            if consumption.sum() > 0:
                category_data[category] = consumption
    
    return pd.DataFrame(category_data) if category_data else None

print("Data loading functions defined.")

In [ ]:
# Load all data sources
print("Loading electricity data...")
electricity_df = load_and_process_electricity_data(electricity_file)

print("\nLoading gas data...")
gas_df = load_and_process_gas_data(gas_file)

print("\nLoading simulated data...")
simulated_df = load_and_process_simulated_data(simulated_file, SCENARIO)

# Combine all data sources
all_categories = {}

if electricity_df is not None:
    all_categories.update({col: electricity_df[col] for col in electricity_df.columns})
    
if gas_df is not None:
    all_categories.update({col: gas_df[col] for col in gas_df.columns})
    
if simulated_df is not None:
    all_categories.update({col: simulated_df[col] for col in simulated_df.columns})

# Create master DataFrame
consumption_df = pd.DataFrame(all_categories)

print(f"\nCombined consumption data:")
print(f"  Shape: {consumption_df.shape}")
print(f"  Categories: {list(consumption_df.columns)}")
print(f"  Date range: {consumption_df.index.min()} to {consumption_df.index.max()}")

# Calculate annual totals
annual_totals = consumption_df.sum().sort_values(ascending=False)
print(f"\nAnnual consumption by category (kWh):")
for category, total in annual_totals.items():
    print(f"  {category}: {total:,.0f} kWh")

total_consumption = annual_totals.sum()
print(f"\nTotal annual consumption: {total_consumption:,.0f} kWh")

In [ ]:
# 1. Annual consumption bar chart
fig, ax = plt.subplots(figsize=(14, 8))

# Get colors for categories
colors = [COLOR_MAP.get(cat, '#BDC3C7') for cat in annual_totals.index]

bars = ax.bar(range(len(annual_totals)), annual_totals.values, color=colors)
ax.set_xticks(range(len(annual_totals)))
ax.set_xticklabels(annual_totals.index, rotation=45, ha='right')
ax.set_ylabel('Annual Consumption (kWh)')
ax.set_title(f'Annual Energy Consumption by End Use\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')

# Add value labels on bars
for bar, value in zip(bars, annual_totals.values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
            f'{value:,.0f}', ha='center', va='bottom', fontsize=9)

# Add percentage labels
for i, (cat, value) in enumerate(annual_totals.items()):
    percentage = (value / total_consumption) * 100
    ax.text(i, value/2, f'{percentage:.1f}%', ha='center', va='center', 
            fontweight='bold', color='white', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\nTotal: {total_consumption:,.0f} kWh/year")

In [ ]:
# 2. Monthly consumption patterns
monthly_consumption = consumption_df.resample('M').sum()
monthly_consumption.index = monthly_consumption.index.strftime('%b')

fig, ax = plt.subplots(figsize=(15, 10))

# Stacked bar chart showing monthly breakdown
bottom = np.zeros(len(monthly_consumption))
colors_list = [COLOR_MAP.get(cat, '#BDC3C7') for cat in consumption_df.columns]

for i, category in enumerate(consumption_df.columns):
    ax.bar(monthly_consumption.index, monthly_consumption[category], 
           bottom=bottom, label=category, color=colors_list[i])
    bottom += monthly_consumption[category]

ax.set_xlabel('Month')
ax.set_ylabel('Monthly Consumption (kWh)')
ax.set_title(f'Monthly Energy Consumption by End Use\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Daily average consumption by month and category
consumption_df['month'] = consumption_df.index.month
consumption_df['hour'] = consumption_df.index.hour

# Calculate daily averages by month for top 6 categories
top_categories = annual_totals.head(6).index
daily_avg_by_month = consumption_df[list(top_categories)].groupby(consumption_df['month']).mean() * 24  # Convert to daily kWh

fig, ax = plt.subplots(figsize=(12, 8))

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

x = np.arange(len(month_names))
width = 0.12  # Width of bars

for i, category in enumerate(top_categories):
    offset = (i - len(top_categories)/2) * width
    color = COLOR_MAP.get(category, '#BDC3C7')
    ax.bar(x + offset, daily_avg_by_month[category], width, 
           label=category, color=color)

ax.set_xlabel('Month')
ax.set_ylabel('Average Daily Consumption (kWh)')
ax.set_title(f'Seasonal Consumption Patterns - Top Categories\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')
ax.set_xticks(x)
ax.set_xticklabels(month_names)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4. Hourly consumption patterns (average by hour of day)
hourly_avg = consumption_df[list(top_categories)].groupby(consumption_df['hour']).mean()

fig, ax = plt.subplots(figsize=(14, 8))

for category in top_categories:
    color = COLOR_MAP.get(category, '#BDC3C7')
    ax.plot(hourly_avg.index, hourly_avg[category], 
            marker='o', linewidth=2, label=category, color=color)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Average Hourly Consumption (kWh)')
ax.set_title(f'Daily Consumption Patterns - Top Categories\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 5. Seasonal heatmap showing consumption patterns
# Create a pivot table for heatmap: month vs hour for total consumption
consumption_df['total'] = consumption_df[list(consumption_df.columns[:-2])].sum(axis=1)  # Exclude month and hour columns
pivot_data = consumption_df.pivot_table(values='total', index='month', columns='hour', aggfunc='mean')

fig, ax = plt.subplots(figsize=(16, 8))

im = ax.imshow(pivot_data.values, cmap='YlOrRd', aspect='auto')

# Set labels
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(12))
ax.set_yticklabels(month_names)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Month')
ax.set_title(f'Total Energy Consumption Heatmap (kWh/hour)\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Average Hourly Consumption (kWh)', rotation=270, labelpad=15)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Peak vs Off-Peak Analysis
# Define peak hours (typically 4 PM - 9 PM for CA)
peak_hours = range(16, 21)  # 4 PM to 9 PM
off_peak_hours = [h for h in range(24) if h not in peak_hours]

consumption_no_time = consumption_df.drop(['month', 'hour', 'total'], axis=1)

peak_consumption = consumption_no_time[consumption_no_time.index.hour.isin(peak_hours)].sum()
off_peak_consumption = consumption_no_time[consumption_no_time.index.hour.isin(off_peak_hours)].sum()

# Create comparison
comparison_data = pd.DataFrame({
    'Peak Hours (4-9 PM)': peak_consumption,
    'Off-Peak Hours': off_peak_consumption
})

# Only show top categories for clarity
comparison_data = comparison_data.loc[top_categories]

fig, ax = plt.subplots(figsize=(12, 8))

comparison_data.plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4'])
ax.set_xlabel('End Use Category')
ax.set_ylabel('Total Consumption (kWh/year)')
ax.set_title(f'Peak vs Off-Peak Consumption by Category\n{COUNTY.replace("-", " ").title()} County - {SCENARIO.replace("_", " ").title()} Scenario')
ax.legend()
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print peak ratios
print("Peak vs Off-Peak Ratios:")
for category in top_categories:
    peak_ratio = peak_consumption[category] / (peak_consumption[category] + off_peak_consumption[category]) * 100
    print(f"  {category}: {peak_ratio:.1f}% during peak hours")

In [ ]:
# 7. Summary statistics and insights
print("=" * 80)
print("ENERGY CONSUMPTION ANALYSIS SUMMARY")
print("=" * 80)
print(f"\nScenario: {SCENARIO.replace('_', ' ').title()}")
print(f"County: {COUNTY.replace('-', ' ').title()}")
print(f"Housing Type: {HOUSING_TYPE.replace('-', ' ').title()}")

print(f"\nTotal Annual Consumption: {total_consumption:,.0f} kWh")
print(f"Average Daily Consumption: {total_consumption/365:.1f} kWh")
print(f"Average Hourly Consumption: {total_consumption/8760:.2f} kWh")

print(f"\nTop 5 Energy Uses:")
for i, (category, consumption) in enumerate(annual_totals.head().items(), 1):
    percentage = (consumption / total_consumption) * 100
    print(f"  {i}. {category}: {consumption:,.0f} kWh ({percentage:.1f}%)")

# Seasonal analysis
monthly_totals = consumption_df.drop(['month', 'hour', 'total'], axis=1).resample('M').sum().sum(axis=1)
peak_month = monthly_totals.idxmax().strftime('%B')
low_month = monthly_totals.idxmin().strftime('%B')
seasonal_ratio = monthly_totals.max() / monthly_totals.min()

print(f"\nSeasonal Patterns:")
print(f"  Peak consumption month: {peak_month} ({monthly_totals.max():,.0f} kWh)")
print(f"  Lowest consumption month: {low_month} ({monthly_totals.min():,.0f} kWh)")
print(f"  Seasonal variation ratio: {seasonal_ratio:.2f}x")

# Daily patterns
hourly_totals = consumption_df.drop(['month', 'hour', 'total'], axis=1).groupby(consumption_df['hour']).mean().sum(axis=1)
peak_hour = hourly_totals.idxmax()
low_hour = hourly_totals.idxmin()

print(f"\nDaily Patterns:")
print(f"  Peak consumption hour: {peak_hour}:00 ({hourly_totals.max():.2f} kWh average)")
print(f"  Lowest consumption hour: {low_hour}:00 ({hourly_totals.min():.2f} kWh average)")
print(f"  Daily variation ratio: {hourly_totals.max() / hourly_totals.min():.2f}x")

print("\n" + "=" * 80)

In [ ]:
# 8. Data export for further analysis
# Save processed data to CSV files
output_dir = Path('analysis_results')
output_dir.mkdir(exist_ok=True)

# Export annual totals
annual_summary = pd.DataFrame({
    'Category': annual_totals.index,
    'Annual_kWh': annual_totals.values,
    'Percentage': (annual_totals.values / total_consumption) * 100
})
annual_file = output_dir / f'annual_consumption_{SCENARIO}_{COUNTY}.csv'
annual_summary.to_csv(annual_file, index=False)

# Export monthly data
monthly_file = output_dir / f'monthly_consumption_{SCENARIO}_{COUNTY}.csv'
monthly_consumption.to_csv(monthly_file)

# Export hourly averages
hourly_file = output_dir / f'hourly_patterns_{SCENARIO}_{COUNTY}.csv'
hourly_avg.to_csv(hourly_file)

print(f"Analysis results exported to:")
print(f"  Annual summary: {annual_file}")
print(f"  Monthly data: {monthly_file}")
print(f"  Hourly patterns: {hourly_file}")

print(f"\nAnalysis complete! Generated {len(plt.get_fignums())} visualizations.")